In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# load processed df
from IPython.utils.capture import capture_output

with capture_output():
    %run balance_over_time.ipynb

In [3]:
features_df

,balance__mean__all,balance__median__all,balance__min__all,balance__max__all,balance__std__all,balance__pct_negative__all,balance__pct_below_100__all,balance__pct_below_500__all,n_days__all,balance__mean__30d,...,cat_39__cat_n__90d,cat_40__cat_n__90d,cat_45__cat_n__90d,cat_46__cat_n__90d,income__total__all,essentials_spend__total__all,discretionary_spend__total__all,essentials__pct_of_income__all,discretionary__pct_of_income__all,DQ_TARGET
prism_consumer_id,,,,,,,,,,,,,,,,,,,,,
0,276.961538,70.09,-1019.10,2732.86,1016.288836,0.475524,0.510490,0.580420,143.0,-497.176071,...,3.0,1.0,0.0,0.0,9340.520508,1718.160034,7332.270020,0.183947,0.784996,0.0
1,1674.533585,1758.35,-123.25,3597.09,1159.524803,0.056604,0.094340,0.301887,106.0,2671.932727,...,0.0,2.0,0.0,0.0,13414.009766,680.210022,11539.479492,0.050709,0.860256,0.0
10,-106.435115,-98.40,-1108.49,929.25,501.726601,0.595420,0.633588,0.839695,131.0,-382.525455,...,0.0,1.0,0.0,4.0,15513.070312,1527.850220,9379.980469,0.098488,0.604650,0.0
100,-3231.228909,-3752.93,-6273.18,802.40,2080.213280,0.963636,0.963636,0.981818,55.0,-4166.195000,...,0.0,0.0,0.0,0.0,24423.531250,18534.431641,200.000000,0.758876,0.008189,0.0
1000,1013.427875,615.39,-22.85,12589.57,1545.044777,0.025000,0.112500,0.437500,80.0,601.327692,...,0.0,0.0,1.0,0.0,58994.343750,17348.220703,0.000000,0.294066,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,49.549474,-44.27,-121.55,776.47,238.842211,0.631579,0.754386,0.929825,57.0,-15.570000,...,2.0,0.0,0.0,0.0,11226.839844,6032.720703,2211.799805,0.537348,0.197010,NaN
9996,172.754000,184.21,32.72,297.21,76.111644,0.000000,0.200000,1.000000,30.0,179.411000,...,0.0,0.0,0.0,0.0,0.030000,331.070038,365.749969,11035.667969,12191.665039,NaN
9997,993.787302,863.25,96.23,2334.40,558.853248,0.000000,0.015873,0.174603,63.0,939.261304,...,6.0,0.0,0.0,0.0,17007.861328,7687.240234,2637.039795,0.451982,0.155048,NaN


In [4]:
'DQ_TARGET' in features_df.columns

True

# Data Preparation

In [5]:
# keep only labeled rows
df = features_df[features_df["DQ_TARGET"].notna()].copy()

X = df.drop(columns=["DQ_TARGET"])
y = df["DQ_TARGET"].astype(int)

## Train Test Split

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Prepare Data
# ---------------------------------------------------------
# Ensure no infinite values
df = features_df.replace([np.inf, -np.inf], np.nan)
df = df[df["DQ_TARGET"].notna()].copy()

X = df.drop(columns=["DQ_TARGET"]).select_dtypes(include=[np.number]).fillna(0)
y = df["DQ_TARGET"].astype(int)

# 2. Create Train / Validation / Test Split (60% / 20% / 20%)
# ---------------------------------------------------------
# First Split: Separate out the Test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Second Split: Separate the remaining 80% into Train (60%) and Val (20%)
# (0.25 of the remaining 80% = 20% of the total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Data Shapes:")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")


Data Shapes:
Train: (6189, 177) | Val: (2064, 177) | Test: (2064, 177)


# Recursive Feature Elimination (RFE)

In [18]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("StandardScaler Done.")

# RFE
model = LogisticRegression(
    solver='liblinear',
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

rfe = RFE(estimator=model, n_features_to_select=30, step=1)

print("Running RFE...")
rfe.fit(X_train_scaled, y_train)


StandardScaler Done.
Running RFE...

Top 30 Features Selected: ['balance__mean__all', 'balance__min__all', 'balance__pct_below_500__all', 'n_days__all', 'balance__mean__30d', 'balance__min__30d', 'cashflow__net__30d', 'cashflow__mean_daily__30d', 'cashflow__volatility__30d', 'n_tx__30d', 'cashflow__net__60d', 'n_tx__60d', 'balance__mean__90d', 'cashflow__mean_daily__90d', 'cashflow__volatility__90d', 'n_days__90d', 'balance__min__180d', 'balance__std__180d', 'cashflow__volatility__180d', 'n_tx__180d', 'n_days__180d', 'tx__n__all', 'tx__std_amount__all', 'tx__max_credit__all', 'cat_6__cat_net_total__all', 'cat_22__cat_net_total__all', 'cat_46__cat_net_total__all', 'cat_6__cat_net_total__90d', 'cat_22__cat_net_total__90d', 'cat_46__cat_net_total__90d']

Training Set AUC: 0.7756
Validation Set AUC:   0.7420
Gap: 0.0336


In [20]:
from sklearn.metrics import roc_auc_score, accuracy_score

# Evaluate Performance (Accuracy & AUC)
# ---------------------------------------------------------
selected_cols = X_train.columns[rfe.support_]
print(f"\nTop {len(selected_cols)} Features Selected: {selected_cols.tolist()}")

# Refit model on selected features
model.fit(X_train_scaled[selected_cols], y_train)

# Predict (Probabilities for AUC, Class Labels for Accuracy)
y_train_prob = model.predict_proba(X_train_scaled[selected_cols])[:, 1]
y_val_prob   = model.predict_proba(X_val_scaled[selected_cols])[:, 1]

y_train_pred = model.predict(X_train_scaled[selected_cols])
y_val_pred   = model.predict(X_val_scaled[selected_cols])

# Calculate Metrics
train_auc = roc_auc_score(y_train, y_train_prob)
val_auc   = roc_auc_score(y_val, y_val_prob)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc   = accuracy_score(y_val, y_val_pred)

print(f"\n--- Performance Report ---")
print(f"Train AUC:      {train_auc:.4f} | Train Accuracy: {train_acc:.4f}")
print(f"Validation AUC: {val_auc:.4f} | Val Accuracy:   {val_acc:.4f}")

# Diagnostic Check
# ---------------------------------------------------------
gap = train_auc - val_auc
print(f"Gap (Train - Val): {gap:.4f}")

if gap > 0.05:
    print("⚠️ Warning: Possible Overfitting. Consider reducing features or increasing regularization (C).")
elif train_auc < 0.60:
    print("⚠️ Warning: Possible Underfitting. Model may be too simple or features are not predictive.")
else:
    print("✅ Model looks healthy.")


Top 30 Features Selected: ['balance__mean__all', 'balance__min__all', 'balance__pct_below_500__all', 'n_days__all', 'balance__mean__30d', 'balance__min__30d', 'cashflow__net__30d', 'cashflow__mean_daily__30d', 'cashflow__volatility__30d', 'n_tx__30d', 'cashflow__net__60d', 'n_tx__60d', 'balance__mean__90d', 'cashflow__mean_daily__90d', 'cashflow__volatility__90d', 'n_days__90d', 'balance__min__180d', 'balance__std__180d', 'cashflow__volatility__180d', 'n_tx__180d', 'n_days__180d', 'tx__n__all', 'tx__std_amount__all', 'tx__max_credit__all', 'cat_6__cat_net_total__all', 'cat_22__cat_net_total__all', 'cat_46__cat_net_total__all', 'cat_6__cat_net_total__90d', 'cat_22__cat_net_total__90d', 'cat_46__cat_net_total__90d']

--- Performance Report ---
Train AUC:      0.7756 | Train Accuracy: 0.6843
Validation AUC: 0.7420 | Val Accuracy:   0.6807
Gap (Train - Val): 0.0336
✅ Model looks healthy.
